# Basic

In [55]:
%load_ext autoreload
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, where src/ lives

import polars as pl
import pickle

import config_prim_fd as config   # still found in decay_with_prim_fd/
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct

D = config.TokenizerParam.max_dist_candidate


In [57]:
df_official = pl.read_parquet(config.BasicConfig.official_release_path, columns=["id", "status"]).unique()
df_concept_hug = pl.read_parquet(config.BasicConfig.concept_path)
df_hug_status = df_concept_hug.join(df_official, on="id", how = "left")
cpt_pre_no_status = df_hug_status.filter(pl.col("status").is_null()).filter(pl.col("concept_type") == "SCT_PRE")["id"].to_list()
cpt_post_no_status = df_hug_status.filter(pl.col("status").is_null()).filter(pl.col("concept_type") == "SCT_POST")["id"].to_list()

post_status = {cpt:"defined" for cpt in cpt_post_no_status}
pre_status = {cpt:"primitive" for cpt in cpt_pre_no_status}
all_status = {**post_status, **pre_status}

df_hug_status = df_hug_status.with_columns(
    pl.col("id")
      .replace_strict(all_status, default=pl.col("status"), return_dtype=pl.Utf8)
      .alias("status")
)

df_hug_status.write_parquet(config.ProcessedGraph.concept_and_status)


In [58]:
graph_fct.build_combined_subgraphs_and_id2label(D)

In [59]:
id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()

In [60]:
len(set(combined_subgraphs.nodes()).intersection(set(pre_status)))

67

In [61]:
import networkx as nx
nx.dag_longest_path_length(combined_subgraphs)   # returns the length (number of edges) of the longest path


30

In [62]:
nx.number_weakly_connected_components(combined_subgraphs)
combined_subgraphs.number_of_nodes()
combined_subgraphs.number_of_edges()

293146

In [ ]:
pl.read_parquet(config.BasicConfig.relation_path)